In [ ]:
import os
import random
import warnings

import numpy as np
from PIL import Image

import torch
from torch.utils.data import Dataset

# Set random seeds for reproducibility
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

# Setting device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
class NpzMedMNIST(Dataset):
    def __init__(self, npz_path, split="train", transform=None):
        data = np.load(npz_path)
        self.images = data[f"{split}_images"]
        self.labels = data[f"{split}_labels"].squeeze()
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img, label = self.images[idx], self.labels[idx]
    
        # Handle different dataset formats
        if img.ndim == 2:  
            img = np.expand_dims(img, -1)
            img = np.repeat(img, 3, axis=-1)
    
        elif img.ndim == 3 and img.shape[-1] == 1:
            img = np.repeat(img, 3, axis=-1)
    
        elif img.ndim == 3 and img.shape[-1] == 3:
            pass  # already RGB
    
        elif img.ndim == 3 and img.shape[-1] == 9:
            H, W, _ = img.shape
            img = img.reshape(H, W, 3, 3).mean(axis=-1).astype(np.uint8)
    
        else:
            raise ValueError(f"Unexpected image shape {img.shape}")
    
        # Convert to PIL
        img = Image.fromarray(img.astype(np.uint8))
    
        # Apply transform to convert PIL → Tensor
        if self.transform:
            img = self.transform(img)
    
        return img, torch.tensor(label, dtype=torch.long)

# Data transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=25),
    #transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), shear=10, scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    #transforms.RandomGrayscale(p=0.1),
    transforms.GaussianBlur(kernel_size=(3, 3), sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
# Data path - Update this to your Kaggle dataset path
Data_path = "/kaggle/input/datasets/maheshnitrkl/aiba-breastmnsit/breastmnist_64.npz"

# Load datasets
train_dataset = NpzMedMNIST(Data_path, split="train", transform=train_transform)
val_dataset = NpzMedMNIST(Data_path, split="val", transform=val_transform)
test_dataset = NpzMedMNIST(Data_path, split="test", transform=test_transform)

In [ ]:
from collections import Counter
import numpy as np

# Function to count class distribution
def print_class_distribution(dataset, split_name):
    
    labels = []

    # Extract labels from dataset
    for i in range(len(dataset)):
        _, label = dataset[i]

        # Handle tensor / numpy labels
        if isinstance(label, np.ndarray):
            label = label.item()
        elif hasattr(label, "item"):
            label = label.item()

        labels.append(label)

    class_counts = Counter(labels)

    print(f"\n{'='*40}")
    print(f"{split_name.upper()} DATASET CLASS DISTRIBUTION")
    print(f"{'='*40}")

    total = len(dataset)

    for cls in sorted(class_counts.keys()):
        count = class_counts[cls]
        percentage = (count / total) * 100

        print(f"Class {cls} : {count} images ({percentage:.2f}%)")

    print(f"\nTotal Images : {total}")


# Print distributions
print_class_distribution(train_dataset, "Train")
print_class_distribution(val_dataset, "Validation")
print_class_distribution(test_dataset, "Test")